# 📖 Notebook 6: RBAC & Security — Locking Down Your Cluster

So far, we've been running everything as a cluster admin — full access to everything.
In a real company, multiple teams share the same cluster. You need to answer:

- **Who** can access **what** resources? (RBAC)
- **Which** pods can talk to **which** other pods? (NetworkPolicies)
- **What** are pods allowed to do on the host? (Pod Security Standards)

This notebook teaches you to build a secure, multi-team Kubernetes environment.

## Learning Objectives

By the end of this notebook, you will be able to:

- Create **namespaces** as isolation boundaries between teams
- Create **ServiceAccounts**, **Roles**, and **RoleBindings** to control API access
- Test permissions with `kubectl auth can-i`
- Apply **NetworkPolicies** to control pod-to-pod traffic
- Enforce **Pod Security Standards** to block dangerous container configurations
- Set **ResourceQuota** and **LimitRange** to prevent resource starvation

## 🛠️ Setup

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

Make sure minikube is running.

In [ ]:
# Verify cluster
!minikube status
!echo '---'
!kubectl get nodes

## 🏢 Step 1: Namespaces as Team Boundaries

A **namespace** is like a virtual cluster inside your cluster. Each team gets their own namespace,
and you can set different permissions, quotas, and policies per namespace.

```
┌─────────────────── Kubernetes Cluster ───────────────────┐
│                                                           │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐      │
│  │  team-alpha  │  │  team-beta   │  │  k8s-lab    │      │
│  │             │  │             │  │             │      │
│  │  pods       │  │  pods       │  │  pods       │      │
│  │  services   │  │  services   │  │  services   │      │
│  │  secrets    │  │  secrets    │  │  secrets    │      │
│  │             │  │             │  │             │      │
│  │  RBAC ✓     │  │  RBAC ✓     │  │  RBAC ✓     │      │
│  │  Quota ✓    │  │  Quota ✓    │  │             │      │
│  │  NetPol ✓   │  │  NetPol ✓   │  │             │      │
│  └─────────────┘  └─────────────┘  └─────────────┘      │
│                                                           │
└───────────────────────────────────────────────────────────┘
```

Let's create two team namespaces:

In [ ]:
# Create team namespaces
!kubectl create namespace team-alpha --dry-run=client -o yaml | kubectl apply -f -
!kubectl create namespace team-beta --dry-run=client -o yaml | kubectl apply -f -

print()
!kubectl get namespaces | grep -E 'NAME|team-|k8s-lab'

## 🔐 Step 2: RBAC — Who Can Do What?

RBAC (Role-Based Access Control) has four building blocks:

```
┌──────────┐      ┌──────────────┐      ┌──────────────┐
│  User /   │─────▶│ RoleBinding   │─────▶│   Role        │
│  Service  │      │ (connects    │      │ (defines     │
│  Account  │      │  who → what) │      │  permissions)│
└──────────┘      └──────────────┘      └──────┬───────┘
                                               │
                                        ┌──────▼───────┐
                                        │  Resources    │
                                        │  pods, svc,   │
                                        │  deployments  │
                                        └──────────────┘
```

- **ServiceAccount**: an identity for pods or automation
- **Role**: a set of permissions (which resources, which verbs) — scoped to one namespace
- **ClusterRole**: same as Role but cluster-wide
- **RoleBinding**: connects a user/ServiceAccount to a Role

### The Golden Rule
**Least privilege**: give the minimum permissions needed. No wildcards. No `cluster-admin`.

### Create a ServiceAccount

A ServiceAccount is like a user account for a pod. By default, Kubernetes auto-mounts a
token into every pod — we'll disable that for security.

In [ ]:
%%writefile /tmp/rbac-sa.yaml
apiVersion: v1
kind: ServiceAccount
metadata:
  name: alpha-app
  namespace: team-alpha
automountServiceAccountToken: false

In [ ]:
!kubectl apply -f /tmp/rbac-sa.yaml
print("\n✅ ServiceAccount created with automountServiceAccountToken: false")

### Create a Role: Read-Only Access

This Role allows listing and getting pods and services — but NOT deleting, creating, or modifying them.

In [ ]:
%%writefile /tmp/rbac-role.yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: pod-reader
  namespace: team-alpha
rules:
  - apiGroups: [""]
    resources: ["pods", "services"]
    verbs: ["get", "list", "watch"]
  - apiGroups: ["apps"]
    resources: ["deployments"]
    verbs: ["get", "list", "watch"]

In [ ]:
!kubectl apply -f /tmp/rbac-role.yaml
print("\n✅ Role 'pod-reader' created in team-alpha")

### Create a RoleBinding: Connect the Dots

Now we bind the `alpha-app` ServiceAccount to the `pod-reader` Role.

In [ ]:
%%writefile /tmp/rbac-binding.yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: alpha-app-reader
  namespace: team-alpha
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: pod-reader
subjects:
  - kind: ServiceAccount
    name: alpha-app
    namespace: team-alpha

In [ ]:
!kubectl apply -f /tmp/rbac-binding.yaml
print("\n✅ RoleBinding created: alpha-app → pod-reader")

### Test Permissions with `kubectl auth can-i`

This is the most useful RBAC debugging command. It answers: "Can this identity do this action?"

In [ ]:
# ✅ Should be ALLOWED — we gave 'list pods' permission
!echo "Can alpha-app list pods in team-alpha?"
!kubectl auth can-i list pods -n team-alpha \
    --as=system:serviceaccount:team-alpha:alpha-app

print()

# ❌ Should be DENIED — we did NOT give 'delete pods' permission
!echo "Can alpha-app delete pods in team-alpha?"
!kubectl auth can-i delete pods -n team-alpha \
    --as=system:serviceaccount:team-alpha:alpha-app

print()

# ❌ Should be DENIED — no access to other namespaces
!echo "Can alpha-app list pods in team-beta?"
!kubectl auth can-i list pods -n team-beta \
    --as=system:serviceaccount:team-alpha:alpha-app

print()

# ❌ Should be DENIED — no access to secrets
!echo "Can alpha-app list secrets in team-alpha?"
!kubectl auth can-i list secrets -n team-alpha \
    --as=system:serviceaccount:team-alpha:alpha-app

**Expected results:**
- `list pods` in team-alpha → **yes** ✅
- `delete pods` in team-alpha → **no** ❌ (read-only role)
- `list pods` in team-beta → **no** ❌ (Role is namespace-scoped)
- `list secrets` in team-alpha → **no** ❌ (secrets not in our Role)

This is **least privilege** in action: the ServiceAccount can only read pods and services in its own namespace.

## 🔒 Step 3: NetworkPolicies — Controlling Pod Traffic

By default, **every pod can talk to every other pod** in the cluster — even across namespaces.
That's dangerous. If an attacker compromises one pod, they can reach everything.

**NetworkPolicies** are firewall rules for pods:

```
Without NetworkPolicy:           With NetworkPolicy:

┌──────┐    ┌──────┐            ┌──────┐    ┌──────┐
│ Pod A │───▶│ Pod B │            │ Pod A │───▶│ Pod B │  ✅ allowed
└──────┘    └──────┘            └──────┘    └──────┘
    │                                │
    ▼                                ▼
┌──────┐                        ┌──────┐
│ Pod C │  ✅ anyone can talk    │ Pod C │  ❌ blocked!
└──────┘                        └──────┘
```

### Best Practice: Default Deny + Explicit Allow

1. Start by denying ALL traffic
2. Then explicitly allow only the traffic your app needs

In [ ]:
# First, deploy a test pod in team-alpha so we can test connectivity
!kubectl run test-pod -n team-alpha --image=busybox:1.36 \
    --restart=Never -- sleep 3600

!kubectl wait --for=condition=ready pod/test-pod -n team-alpha --timeout=30s
print("\n✅ Test pod running in team-alpha")

In [ ]:
# Before NetworkPolicy: can test-pod reach pods in k8s-lab?
!echo "Testing connectivity to api-gateway in k8s-lab namespace..."
!kubectl exec -n team-alpha test-pod -- \
    wget -qO- --timeout=3 http://api-gateway.k8s-lab:8000/health 2>&1 || echo "Connection failed"

print("\n☝️ Without NetworkPolicy, cross-namespace traffic is ALLOWED")

### Apply Default-Deny Policy

This policy blocks ALL ingress and egress traffic for every pod in the namespace.

In [ ]:
%%writefile /tmp/default-deny.yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-all
  namespace: team-alpha
spec:
  podSelector: {}
  policyTypes:
    - Ingress
    - Egress

In [ ]:
!kubectl apply -f /tmp/default-deny.yaml

print("\n🔒 Default-deny applied to team-alpha")
print("Now let's test — the same request should FAIL:")
print()

# After NetworkPolicy: same test should now fail
!kubectl exec -n team-alpha test-pod -- \
    wget -qO- --timeout=3 http://api-gateway.k8s-lab:8000/health 2>&1 || echo "❌ Connection blocked (as expected!)"

### Allow Specific Traffic

Now let's allow DNS (so pods can resolve service names) and allow egress to k8s-lab's api-gateway.

In [ ]:
%%writefile /tmp/allow-dns-and-gateway.yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-dns
  namespace: team-alpha
spec:
  podSelector: {}
  policyTypes:
    - Egress
  egress:
    - ports:
        - port: 53
          protocol: UDP
        - port: 53
          protocol: TCP
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-to-gateway
  namespace: team-alpha
spec:
  podSelector: {}
  policyTypes:
    - Egress
  egress:
    - to:
        - namespaceSelector:
            matchLabels:
              kubernetes.io/metadata.name: k8s-lab
          podSelector:
            matchLabels:
              app: api-gateway
      ports:
        - port: 8000

In [ ]:
!kubectl apply -f /tmp/allow-dns-and-gateway.yaml

import time
time.sleep(2)

# Now api-gateway should be reachable but user-service should NOT
print("\nTest 1: api-gateway (should succeed):")
!kubectl exec -n team-alpha test-pod -- \
    wget -qO- --timeout=5 http://api-gateway.k8s-lab:8000/health 2>&1 || echo "Connection failed"

print("\nTest 2: user-service directly (should fail):")
!kubectl exec -n team-alpha test-pod -- \
    wget -qO- --timeout=3 http://user-service.k8s-lab:8001/health 2>&1 || echo "❌ Connection blocked (as expected!)"

**Result**: The NetworkPolicy acts as a firewall — team-alpha pods can reach the api-gateway
but NOT the backend services directly. This is the principle of **least privilege** applied to networking.

## 🛡️ Step 4: Pod Security Standards

Kubernetes has three built-in security profiles that prevent pods from doing dangerous things:

| Level | What It Blocks |
|-------|----------------|
| **Privileged** | Nothing — allows everything (development only) |
| **Baseline** | Blocks the most dangerous options: hostNetwork, hostPID, privileged containers |
| **Restricted** | Blocks everything above + requires non-root, read-only filesystem, drops all capabilities |

You apply them using **labels** on namespaces:
- `enforce` → rejects pods that violate the policy
- `warn` → allows but prints a warning
- `audit` → allows but logs in the audit log

In [ ]:
# Apply Pod Security Standards to team-alpha
# enforce=baseline: block dangerous pods
# warn=restricted: show warnings for non-optimal pods
!kubectl label namespace team-alpha \
    pod-security.kubernetes.io/enforce=baseline \
    pod-security.kubernetes.io/warn=restricted \
    --overwrite

print("\n✅ Pod Security Standards applied to team-alpha")
!kubectl get namespace team-alpha --show-labels | grep -o 'pod-security[^ ]*'

In [ ]:
# Try to create a privileged pod — this should be REJECTED
!echo "Attempting to create a privileged pod..."
!kubectl run evil-pod -n team-alpha --image=nginx \
    --overrides='{"spec":{"containers":[{"name":"evil","image":"nginx","securityContext":{"privileged":true}}]}}' \
    2>&1

print("\n☝️ The pod was REJECTED because privileged:true violates the 'baseline' policy.")

In [ ]:
# A normal pod should still work fine
!kubectl run good-pod -n team-alpha --image=nginx:1.27 2>&1
print("\n✅ Normal (non-privileged) pod created successfully.")

# Clean up
!kubectl delete pod good-pod -n team-alpha --ignore-not-found

## 📊 Step 5: ResourceQuota and LimitRange

Even with RBAC and NetworkPolicies, a team could still consume all cluster resources
and starve other teams. **ResourceQuota** prevents that.

- **ResourceQuota**: hard limits on total CPU, memory, and pod count per namespace
- **LimitRange**: default resource requests/limits for individual pods

In [ ]:
%%writefile /tmp/quota.yaml
apiVersion: v1
kind: ResourceQuota
metadata:
  name: team-alpha-quota
  namespace: team-alpha
spec:
  hard:
    requests.cpu: "2"
    requests.memory: 2Gi
    limits.cpu: "4"
    limits.memory: 4Gi
    pods: "20"
---
apiVersion: v1
kind: LimitRange
metadata:
  name: team-alpha-limits
  namespace: team-alpha
spec:
  limits:
    - default:
        cpu: 200m
        memory: 256Mi
      defaultRequest:
        cpu: 100m
        memory: 128Mi
      type: Container

In [ ]:
!kubectl apply -f /tmp/quota.yaml

print("\n✅ ResourceQuota and LimitRange applied to team-alpha")
print("\n--- ResourceQuota ---")
!kubectl describe resourcequota team-alpha-quota -n team-alpha

print("\n--- LimitRange ---")
!kubectl describe limitrange team-alpha-limits -n team-alpha

In [ ]:
# When you create a pod without specifying resources, LimitRange adds defaults:
!kubectl run auto-limits-pod -n team-alpha --image=nginx:1.27
!kubectl wait --for=condition=ready pod/auto-limits-pod -n team-alpha --timeout=30s

print("\nNotice the auto-assigned resource requests/limits:")
!kubectl get pod auto-limits-pod -n team-alpha -o jsonpath='{.spec.containers[0].resources}' | python3 -m json.tool

# Clean up
!kubectl delete pod auto-limits-pod -n team-alpha --ignore-not-found

## 🧹 Clean Up

In [ ]:
# Delete the test pod
!kubectl delete pod test-pod -n team-alpha --ignore-not-found

# Remove NetworkPolicies
!kubectl delete networkpolicy --all -n team-alpha

# Clean up temp files
!rm -f /tmp/rbac-sa.yaml /tmp/rbac-role.yaml /tmp/rbac-binding.yaml
!rm -f /tmp/default-deny.yaml /tmp/allow-dns-and-gateway.yaml /tmp/quota.yaml

# Keep namespaces for later labs
print("✅ Cleaned up! (namespaces team-alpha and team-beta kept for later labs)")

## 🎓 What You Learned

In this notebook you:

1. **Created namespaces** as isolation boundaries for different teams
2. **Set up RBAC** with ServiceAccount → Role → RoleBinding to grant read-only access
3. **Tested permissions** with `kubectl auth can-i` — confirmed least-privilege works
4. **Applied NetworkPolicies** — default-deny + explicit allow for specific traffic
5. **Enforced Pod Security Standards** — blocked privileged containers at the namespace level
6. **Set ResourceQuota and LimitRange** — prevented resource starvation between teams

### Key Takeaways

- **Namespaces** are the primary isolation boundary in Kubernetes
- **RBAC** controls API access — always use least privilege, avoid wildcards and `cluster-admin`
- **NetworkPolicies** control network traffic — always start with default-deny
- **Pod Security Standards** prevent dangerous container configurations — use `baseline` at minimum
- **ResourceQuota** prevents one team from consuming all cluster resources
- Test everything with `kubectl auth can-i` — it's your best RBAC debugging tool

### Security Checklist for Production Namespaces

| ✅ | Item |
|----|------|
| ☐ | Dedicated namespace per team |
| ☐ | RBAC Role + RoleBinding (no ClusterRoleBindings) |
| ☐ | ServiceAccount with automountServiceAccountToken: false |
| ☐ | NetworkPolicy default-deny + explicit allows |
| ☐ | Pod Security Standards: baseline (enforce) + restricted (warn) |
| ☐ | ResourceQuota for CPU, memory, pods |
| ☐ | LimitRange for default requests/limits |

### Next Steps

In **Notebook 07** you'll learn GitOps with ArgoCD — deploying and managing your apps
through Git instead of running `kubectl apply` manually.